# Run the VDocRAG demo

Launches the actual Gradio app (`app.py`) with the real, GPU-loaded retriever/
generator. Requires Steps 1–3 (`00_smoke_test.ipynb`, `01_wrapper_test.ipynb`)
to have already confirmed this configuration works on your hardware — this
notebook doesn't re-diagnose problems, it just runs the app.

If this notebook fails somewhere Steps 1–3 didn't, that's a real gap between
the tested path and `app.py`'s actual usage — worth reporting, not chasing
cell-by-cell fixes here first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['HF_HOME'] = '/content/drive/MyDrive/vdocrag-project/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

In [ ]:
!apt-get install -y poppler-utils -q

!pip install -q --upgrade pip
# pinned to the last 4.x release before transformers' 5.0 major version --
# see docs/implementation_plan.md Section 4.6b-4.6f for why this exact version.
!pip install -q transformers==4.57.3 accelerate bitsandbytes peft pillow
!pip install -q pdf2image faiss-cpu gradio pandas

# NTT's package installed by URL every session, never vendored -- see docs/licenses.md
!pip install -q git+https://github.com/nttmdlab-nlp/VDocRAG.git

!rm -rf /content/repo
!git clone https://github.com/thejainamjain/vdocrag-project.git /content/repo
import sys
sys.path.insert(0, '/content/repo')

## Load the app

This one cell does everything Steps 1–3 verified piece by piece: loads the
quantized base model with the confirmed `eager` + `num_crops=4` config, attaches
both LoRA adapters with shared-base hot-swap, wraps them in the retriever/
generator classes, and builds the Gradio Blocks UI around them. Expect a few
minutes on first run (model download); fast on later runs (Drive-cached weights).

In [ ]:
from app import main
demo = main()

## Launch

Kept as its own cell (separate from the load step above) so the notebook can
be re-launched — e.g. after a Gradio crash or to pick up a UI tweak in `app.py`
— without reloading the multi-GB model each time. `share=True` gives a public
URL usable from your Mac's browser; this cell blocks until you interrupt it.

In [ ]:
demo.launch(share=True, debug=True)